# AGILAB worker class paths in Colab

This notebook separates worker-class resolution from the app launch flow.

- `execution_pandas_project` resolves to `ExecutionPandasWorker` and the `PandasWorker` family.
- `flight_project` resolves to `FlightWorker` and the `PolarsWorker` family.
- `uav_relay_queue_project` is shown through `AgiEnv` path resolution, while `DagWorker` is shown separately from the shared core.

It installs the published package, clones the AGILAB repository to get the built-in app sources, and prints the worker source path and class lineage for each case.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/examples/notebook_quickstart/agi_core_colab_worker_paths.ipynb)


In [ ]:
!pip install -q --upgrade agilab
!git clone --depth 1 https://github.com/ThalesGroup/agilab.git /content/agilab

from pathlib import Path
import sys

REPO_ROOT = Path("/content/agilab")
BUILTIN_ROOT = REPO_ROOT / "src" / "agilab" / "apps" / "builtin"
for path in [
    BUILTIN_ROOT / "execution_pandas_project" / "src",
    BUILTIN_ROOT / "flight_project" / "src",
]:
    sys.path.insert(0, str(path))

print("Repo root:", REPO_ROOT)
print("Builtin root:", BUILTIN_ROOT)
print("execution_pandas_project source:", (BUILTIN_ROOT / "execution_pandas_project" / "src").is_dir())
print("flight_project source:", (BUILTIN_ROOT / "flight_project" / "src").is_dir())
print("uav_relay_queue_project source:", (BUILTIN_ROOT / "uav_relay_queue_project" / "src").is_dir())


In [ ]:
from agi_env import AgiEnv
from agi_node.dag_worker import DagWorker
from agi_node.pandas_worker import PandasWorker
from agi_node.polars_worker import PolarsWorker

from execution_pandas_worker.execution_pandas_worker import ExecutionPandasWorker
from flight_worker.flight_worker import FlightWorker

def describe_app(app_name: str, imported_cls=None):
    env = AgiEnv(apps_path=BUILTIN_ROOT, app=app_name, verbose=0)
    print(f"\n=== {app_name} ===")
    print("target worker class:", getattr(env, "target_worker_class", None))
    print("worker source:", getattr(env, "worker_path", None))
    print("resolved worker target:", getattr(env, "target_worker", None))
    if imported_cls is not None:
        print("imported class:", imported_cls.__module__ + "." + imported_cls.__name__)
        print("MRO:", " -> ".join(cls.__name__ for cls in imported_cls.mro()[:4]))
    return env

describe_app("execution_pandas_project", ExecutionPandasWorker)
describe_app("flight_project", FlightWorker)
describe_app("uav_relay_queue_project")

print("\n=== shared DagWorker ===")
print("imported class:", DagWorker.__module__ + "." + DagWorker.__name__)
print("MRO:", " -> ".join(cls.__name__ for cls in DagWorker.mro()[:4]))
print("base worker families:", PandasWorker.__name__, "/", PolarsWorker.__name__)


## What this proves

- `agilab` can be installed directly from PyPI in a notebook runtime.
- The built-in app sources can be fetched by cloning the repository inside Colab.
- `AgiEnv` resolves the worker source path for each built-in app.
- `execution_pandas_project` binds to `ExecutionPandasWorker`, whose base family is `PandasWorker`.
- `flight_project` binds to `FlightWorker`, whose base family is `PolarsWorker`.
- `DagWorker` is available as the shared DAG execution base class.
- `uav_relay_queue_project` stays visible as a resolved app worker path, without pretending it is the same thing as `DagWorker`.

## Next step

After this class-path check works, use the launch notebooks if you want to run the apps end to end.
